📌 PROJETOS FUTUROS

1. Bot atual (terminar)
2. Bot com API Binance
3. Assistente de código (meu próprio helper)
4. [outros...]

In [1]:
#bibliotecas
import time
import datetime
import re
import random
import unicodedata
from confing import Config
from times_robo import times
from selenium.common.exceptions import TimeoutException
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    StaleElementReferenceException
)

In [2]:
#painel de controle
resposta = input("Nivel de log (0=off, 1=info, 2=debug): ").strip()

if resposta.isdigit():
    LOG_LEVEL = int(resposta)
else:
    LOG_LEVEL = 0  # padrão


# ===== CORES ANSI =====
CORES = {
    "RESET": "\033[0m",
    "OCUTOS": "\033[90m",  # cinza
    "ERRO": "\033[91m",    # vermelho
    "OK": "\033[92m",      # verde
    "ALERTA": "\033[93m",  # amarelo
    "INFO": "\033[96m",   # ciano claro/neon
}

TIPOS_NIVEL = {
    "OCUTOS": 2,
    "INFO": 1,
    "OK": 1,
    "ALERTA": 1,
    "ERRO": 1}

In [3]:
# dados do usuario
user = Config.USERNAME
senha = Config.PASSWORD
url2 = "https://esportesdasorte.bet.br/ptb/bet/main"
url1 = "https://www.flashscore.com.br/"
ZENOH = "\033[93m⚡ ZEN'OH ⚡"
XPATH_FINAL = '//h6[@class="footer-title" and contains(text(), "Suporte")]'

In [4]:
#ferramentas auxiliares

# verificar_time
def verificar_time(nome):
    base = (nome)

    if base in [(n) for n in times]:
        debug(f"🔎 achei {base} na lista",tipo="OK")
    else:
        debug(f"❌ Nao achei o {base}",tipo="ALERTA")

#testar xpath
def testar_xpath(navegador, xpath):

    try:

        elementos = navegador.find_elements(By.XPATH, xpath)

        total = len(elementos)

        debug(f"🔎 XPath: {xpath}", tipo="INFO")
        debug(f"📦 Total encontrado: {total}", tipo="OK")

        if total == 0:
            debug("❌ Nenhum elemento encontrado", tipo="ERRO")
            return

        for i, el in enumerate(elementos):

            try:

                texto = el.text.strip()

                print("\n-------------------")
                print(f"Índice : {i}")
                print(f"Texto  : {repr(texto)}")
                print(f"Tag    : {el.tag_name}")
                print(f"Classe : {el.get_attribute('class')}")

            except Exception as e:
                debug(f"⚠ Erro no elemento {i}: {e}", tipo="ALERTA")

    except Exception as e:
        debug(f"❌ Erro geral: {e}", tipo="ERRO")

In [5]:
# todas as fuçoes

# ================= ferramentas =================
#funçao
def esperar(min=1.0, max=9.0): 
    time.sleep(random.uniform(min, max))

#funçao (BOT)
def mover_mouse(driver, elemento):
    try:
        ActionChains(driver).move_to_element(elemento).perform()
        esperar(0.2, 0.8)
    except:
        pass

#funçao (BOT)
def scroll_humano(
    driver,
    direcao="descer",
    movimentos=(8, 12)
):

    body = driver.find_element(By.TAG_NAME, "body")

    # ========= DESCER =========
    if direcao == "descer":

        for _ in range(random.randint(*movimentos)):

            final = driver.find_elements(By.XPATH, XPATH_FINAL)

            if final:
                debug("✅ Final encontrado", tipo="OCUTOS")
                break

            body.send_keys(Keys.PAGE_DOWN)
            esperar(0.2, 0.7)

    # ========= SUBIR =========
    elif direcao == "subir":

        for _ in range(random.randint(*movimentos)):

            topo = driver.execute_script(
                "return window.pageYOffset;"
            )

            if topo <= 0:
                debug("🏠 Topo encontrado", tipo="OCUTOS")
                break

            body.send_keys(Keys.PAGE_UP)
            esperar(0.2, 0.7)

#auxiliar=1
def debug(*msg, nivel=None, tipo="INFO"):
    # se não passou nível, usa o padrão do tipo
    if nivel is None:
        nivel = TIPOS_NIVEL.get(tipo, 1)

    if LOG_LEVEL < nivel:
        return

    agora = datetime.datetime.now().strftime("%H:%M:%S")
    texto = " ".join(str(m) for m in msg)
    mensagem = f"[{agora}] [{tipo}] {texto}"

    cor = CORES.get(tipo, CORES["RESET"])
    print(f"{cor}{mensagem}{CORES['RESET']}")

#auxiliar=2
def debug_linha(*msg, nivel=None, tipo="INFO"):
    # usa o mesmo padrão do debug()
    if nivel is None:
        nivel = TIPOS_NIVEL.get(tipo, 1)

    if LOG_LEVEL < nivel:
        return

    agora = datetime.datetime.now().strftime("%H:%M:%S")
    texto = " ".join(str(m) for m in msg)
    mensagem = f"[{agora}] [{tipo}] {texto}"

    cor = CORES.get(tipo, CORES["RESET"])

    # limpa linha + reescreve
    print("\r" + " " * 120, end="")
    print(f"\r{cor}{mensagem}{CORES['RESET']}", end="", flush=True)

#funçao (BOT)
def criar_wait(driver, tempo=10):
    return WebDriverWait(driver, tempo)

#funçao (BOT)
def separador(pula=0):
    print()
    print("═" * 60)
    print()
    for _ in range(pula):
        print()

# ================= CONFIG =================
#funçao  (BOT)
def iniciar_driver():
    options = Options()

    # PERFIL FIXO (IMPORTANTE)
    options.add_argument(r"--user-data-dir=C:\chrome-bot")

    # ================= ANTI-DETECÇÃO =================
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    # ================= PERFORMANCE / ESTABILIDADE =================
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-infobars")

    # ================= TELA =================
    options.add_argument("--window-size=1366,768")

    
    # ================= INICIAR =================
    driver = webdriver.Chrome(options=options)

    # Garantir tamanho consistente
    driver.maximize_window()

    # Remover flag webdriver
    driver.execute_script("""
        Object.defineProperty(navigator, 'webdriver', {
            get: () => undefined
        })
    """)


    return driver

#funçao  (BOT)
def abrir_sites(driver, *urls):

    separador()

    primeira_url = True
    num = 1

    for url in urls:

        try:

            # PRIMEIRA ABA
            if primeira_url:

                debug(f"🌐 Abrindo página n-{num} >>>>: {url}", tipo="INFO")

                driver.get(url)

                if esperar_pagina(driver, tentativas=3):

                    debug("✅ Página carregada!", tipo="OK")

                else:

                    debug("⚠️ Página demorou carregar", tipo="ERRO")

                primeira_url = False
                continue

            # NOVAS ABAS
            separador()

            driver.execute_script(f"window.open('{url}');")

            # pega última aba aberta
            driver.switch_to.window(driver.window_handles[-1])

            num += 1

            debug(f"🌐 Abrindo página n-{num} >>>>: {url}", tipo="INFO")

            if esperar_pagina(driver, tentativas=3):

                debug("✅ Página carregada!", tipo="OK")

            else:

                debug("⚠️ Página demorou carregar", tipo="ERRO")

            separador()

        except Exception as e:

            debug(f"❌ Erro ao abrir nova aba: {e}", tipo="ERRO")
        
#funçao (BOT)
def ir_para_aba(driver, numero_aba):
    abas = driver.window_handles

    try:
        driver.switch_to.window(abas[numero_aba - 1])

        debug(f"🔄 Mudou para aba {numero_aba}")
        debug("🌐 URL atual:", driver.current_url)

    except IndexError:
        debug(f"❌ Aba {numero_aba} não existe",tipo="alerta")

#funçao (BOT)
def esperar_pagina(driver, tentativas=3):

    for tentativa in range(tentativas):

        try:

            WebDriverWait(driver, 30).until(
                lambda d: d.execute_script(
                    "return document.readyState"
                ) == "complete"
            )

            return True

        except TimeoutException:

            debug(
                f"⚠️ Tentativa {tentativa+1} falhou",
                tipo="ERRO"
            )

            driver.refresh()

    return False


# ================= COOKIES =================
#funçao (BOT)
def limpar_popups(driver):
    for _ in range(5):
        esperar(1,5)
        if not limpar_popups_hard(driver):
            break

#auxiliar

def limpar_popups_hard(driver):
    try:
        driver.execute_script("""
        // 🔥 1. remover overlays pesados
        document.querySelectorAll('*').forEach(el => {
            let style = window.getComputedStyle(el);

            if (
                style.position === 'fixed' &&
                parseInt(style.zIndex) > 999
            ) {
                el.remove();
            }
        });

        // 🔥 2. remover TODOS os iframes suspeitos
        document.querySelectorAll('iframe').forEach(f => f.remove());

        // 🔥 3. liberar scroll
        document.body.style.overflow = 'auto';
        document.documentElement.style.overflow = 'auto';

        // 🔥 4. remover backdrop (muito comum)
        document.querySelectorAll('[class*="backdrop"], [class*="overlay"]').forEach(el => el.remove());

        // 🔥 5. parar possíveis funções de popup
        window.open = () => null;

        // 🔥 6. bloquear reaparecimento FORTE
        if (!window.killPopupObserver) {
            window.killPopupObserver = new MutationObserver(() => {
                document.querySelectorAll('*').forEach(el => {
                    let style = window.getComputedStyle(el);

                    if (
                        style.position === 'fixed' &&
                        parseInt(style.zIndex) > 999
                    ) {
                        el.remove();
                    }
                });

                document.querySelectorAll('iframe').forEach(f => f.remove());

                document.body.style.overflow = 'auto';
            });

            window.killPopupObserver.observe(document.documentElement, {
                childList: true,
                subtree: true
            });
        }
        """)

        debug("💀 Popup exterminado (modo insano)",tipo="OCUTOS")
        return True

    except Exception as e:
        debug(f"❌ Erro:, {e}",tipo="OCUTOS")
        return False
  

# ================= LOGIN =================
#funçao (BOT)
def login_se_necessario(driver):
    debug("⏳ Verificando status do login... aguarde", tipo="INFO")
    if not esta_logado(driver):
        fazer_login(driver)
        debug("🔐 Login realizado com sucesso!", tipo="OK")
    else:
        debug("🚩 já estava longado!!!",tipo="OK")

#auxiliar
def esta_logado(driver):
    try:
        WebDriverWait(driver, 30).until(
            EC.presence_of_element_located((By.CLASS_NAME, "desktop-balance-text"))
        )
        return True
    except:
        return False
#auxiliar
def fazer_login(driver):
    debug("🔐 Fazendo login...",tipo="OK")

    campo_login = WebDriverWait(driver, 20).until(
        EC.presence_of_element_located((By.ID, "unifiedLogin"))
    )
    campo_login.clear()
    campo_login.send_keys(user)

    campo_senha = driver.find_element(By.ID, "password-login")
    campo_senha.clear()
    campo_senha.send_keys(senha)

    botao_login = driver.find_element(By.CSS_SELECTOR, 'button[data-test="header-login-button"]')
    botao_login.click()

    esperar(2, 4)


# ================ monitoramento =============
#funçao BOT
def favoritar(driver, times):

    try:
        debug("1️⃣ processar_lista_times",tipo="OCUTOS")
        processar_lista_times(driver, times)

        debug("2️⃣ clicar_favoritos",tipo="OCUTOS")
        clicar_favoritos(driver)

        debug("✅ favoritos Finalizado!!!", tipo="OK")
        separador()
    except Exception as erro:
        debug(f"💥 Erro: {erro}", tipo="ALERTA")

#funçao BOT
def limpar_favoritos(driver):
    debug("⏳ Limpando favoritos ... aguarde", tipo="INFO")
    while True:
        alterou = False
        jogos = driver.find_elements(By.XPATH, '//div[contains(@id,"g_")]')

        for i in range(len(jogos)):
            try:
                jogos = driver.find_elements(By.XPATH, '//div[contains(@id,"g_")]')
                jogo = jogos[i]

                if processar_jogo(driver, jogo):
                    alterou = True
                    break

            except:
                continue

        if not alterou:
            print()
            debug("✅ Limpeza finalizada!", tipo="OK")
            break

#funçao BOT
def pegar_times_hoje(navegador, times):

    jogos_da_msm_lista = []
    times2 = []

    # 🔥 limpa as chaves do dicionário
    times_limpos = {
        limpar_nome(k): v
        for k, v in times.items()
    }

    jogos = navegador.find_elements(
        By.XPATH,
        '//div[@data-event-row="true"]'
    )

    debug(f"📦 Jogos encontrados: {len(jogos)}", tipo="OCUTOS")

    for i, jogo in enumerate(jogos):

        try:

            elementos_times = jogo.find_elements(
                By.XPATH,
                './/span[@data-testid="wcl-scores-simple-text-01"]'
            )

            if len(elementos_times) < 2:
                debug(f"⚠ Jogo {i} sem times suficientes", tipo="ALERTA")
                continue

            casa = limpar_nome(elementos_times[0].text)
            fora = limpar_nome(elementos_times[1].text)

            

            debug(f"🏠 Casa: {casa}", tipo="OCUTOS")
            debug(f"🛫 Fora: {fora}", tipo="OCUTOS")

    
            # 🔥 os dois times estão na lista
            if casa in times_limpos and fora in times_limpos:

                debug("🔥 jogo interno encontrado", tipo="OCUTOS")

                jogos_da_msm_lista.append({
                    "casa": casa,
                    "fora": fora
                })

            # 🔥 apenas um dos times está
            elif casa in times_limpos or fora in times_limpos:

                debug("⚠ jogo externo encontrado", tipo="OCUTOS")

                if casa in times_limpos:
                    times2.append(times_limpos[casa])

                if fora in times_limpos:
                    times2.append(times_limpos[fora])


        except Exception as e:
            debug(f"❌ Erro no jogo {i}: {e}", tipo="ERRO")

    debug(
        f"📦 Jogos internos encontrados: {len(jogos_da_msm_lista)}",
        tipo="INFO"
    )

    debug(
        f"🌎 Jogos externos encontrados: {len(times2)}",
        tipo="INFO"
    )

    return {
        "jogos_internos": jogos_da_msm_lista,
        "jogos_externos": times2
    }


# ================= AUXILIARES =================
def limpar_nome(nome):

    # 🔥 minúsculo
    nome = nome.lower()

    # 🔥 remove texto entre ()
    nome = re.sub(r"\(.*?\)", "", nome)

    # 🔥 remove acentos
    nome = unicodedata.normalize('NFD', nome)
    nome = nome.encode('ascii', 'ignore').decode('utf-8')

    # 🔥 remove espaços duplicados
    nome = re.sub(r"\s+", " ", nome)

    # 🔥 remove espaços do começo/fim
    nome = nome.strip()

    return nome

def clicar_favoritos(driver):
    try:
        separador()
        botao = WebDriverWait(driver, 30).until(
            EC.element_to_be_clickable(
                (By.XPATH, '//a[contains(@href, "favoritos")]')
            )
        )

        botao.click()
        debug("⭐ Indo para favoritos...", tipo="INFO")

        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located(
                (By.XPATH, '//a[@aria-current="page" and contains(@href,"favoritos")]')
            )
        )

    except Exception as e:
        debug(f"❌ Erro ao clicar nos favoritos: {e}", tipo="ERRO")

def encontrar_time_exato(driver, nome_time, timeout=10):
    # 🔥 tentativa 1 (img)
    try:
        return WebDriverWait(driver, timeout).until(
            EC.visibility_of_element_located(
                (By.XPATH, f'//img[contains(@alt,"{nome_time}")]')
            )
        )
    except:
        pass

    # 🔥 tentativa 2 (texto)
    try:
        return WebDriverWait(driver, timeout).until(
            EC.visibility_of_element_located(
                (By.XPATH,
                 f'//*[contains(translate(text(),"ABCDEFGHIJKLMNOPQRSTUVWXYZÁÉÍÓÚÃÕÇ","abcdefghijklmnopqrstuvwxyzáéíóúãõç"), "{base}")]')
            )
        )
    except:
        return None

def scroll_se_precisar(driver, elemento):
    if elemento is None:
        return

    try:
        visivel = driver.execute_script("""
            const el = arguments[0];
            const rect = el.getBoundingClientRect();
            return rect.top >= 0 && rect.bottom <= window.innerHeight;
        """, elemento)

        if not visivel:
            driver.execute_script(
                "arguments[0].scrollIntoView({block: 'center'});",
                elemento
            )
    except:
        pass

def pegar_container(driver, imagem, timeout=20):
    if imagem is None:
        return None

    return WebDriverWait(driver, timeout).until(
        lambda d: imagem.find_element(
            By.XPATH,
            './ancestor::div[contains(@id,"g_")]'
        )
    )

def pegar_botao_favorito(container, driver):
    if container is None:
        return None

    seletor = 'button[data-testid*="favorite"]'

    try:
        # 🚀 tentativa rápida
        return WebDriverWait(driver, 5).until(
            lambda d: container.find_element(By.CSS_SELECTOR, seletor)
        )

    except:
        debug("⚠ Tentando novamente...", tipo="OCUTOS")

        try:
            # segunda tentativa mais segura
            return WebDriverWait(driver, 5).until(
                lambda d: container.find_element(By.CSS_SELECTOR, seletor)
            )
        except:
            debug("❌ Botão de favorito não encontrado", tipo="ERRO")
            return None

def aguardar_botao_clicavel(driver, timeout=20):
    WebDriverWait(driver, timeout).until(
        EC.element_to_be_clickable(
            (By.CSS_SELECTOR, 'button[data-testid*="favorite"]')
        )
    )

def favoritar_se_preciso(botao):
    if botao is None:
        return None

    estado = botao.get_attribute("aria-pressed")

    if estado == "false":
        botao.click()
        esperar(0.2, 0.3)
        debug("✅ Time encontrado! ⭐Favoritado!!", tipo="OK")
        return True
    else:
        debug("⭐ Já estava favoritado!", tipo="ALERTA")
        return False

# ================= PROCESSAMENTO =================
# auxiliar
def processar_time(driver, nome_time):
    separador()
    debug_linha("🔍 Procurando o time: ➡➡➡", nome_time)
    print()
    try:
        imagem = encontrar_time_exato(driver, nome_time)
        
        # ❌  AQUI decide se joga ou não
        if not imagem:
            esperar(0.1,0.5)
            debug(f"⚠ O time {nome_time} não joga hoje 💥 Próximo", tipo="ERRO")
            separador()
            return

      
        # 🔥 scroll seguro (NO ELEMENTO)
        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});", imagem
        )
        time.sleep(0.3)

        container = pegar_container(driver, imagem)
        if container is None:
            debug("⚠ container não encontrado", tipo="ERRO")
            return

        botao = pegar_botao_favorito(container,driver)
        if botao is None:
            debug("⚠ botão não encontrado", tipo="ERRO")
            return

        aguardar_botao_clicavel(driver)
        favoritar_se_preciso(botao)
        separador()

    except Exception as e:
        print()
        debug(f"❌ Erro ao processar {nome_time}: {e}", tipo="OCUTOS")
# auxiliar
def processar_lista_times(driver, times):
    lista_times = list(times.keys())

    for nome_time in lista_times:
        processar_time(driver, nome_time)
        esperar(0.1,2.0)

# ================= FILTRO =================

def eh_time_banido(nome):
    return bool(padrao.search(nome))

def processar_jogo(driver, jogo):
    try:
        times = jogo.find_elements(
            By.XPATH, './/span[@data-testid="wcl-scores-simple-text-01"]'
        )

        if len(times) < 2:
            return False

        casa = times[0].text.strip()
        fora = times[1].text.strip()

        if eh_time_banido(casa) or eh_time_banido(fora):
            botoes = jogo.find_elements(
                By.XPATH, './/button[@data-testid="wcl-favorite-active"]'
            )

            if botoes:
                botao = botoes[0]

                driver.execute_script(
                    "arguments[0].scrollIntoView({block: 'center'});", botao
                )

                time.sleep(0.2)

                try:
                    botao.click()
                except:
                    driver.execute_script("arguments[0].click();", botao)

                debug(f"⭐ Removido: {casa} vs {fora}", tipo="OK")
                return True

        return False

    except:
        return False


padrao = re.compile(
    r'('
    r'\bu-?\s?21\b|'
    r'\bu-?\s?20\b|'
    r'\bu-?\s?19\b|'
    r'\bu-?\s?17\b|'
    r'\bsub[-\s]?21\b|'
    r'\bsub[-\s]?20\b|'
    r'\bsub[-\s]?19\b|'
    r'\bsub[-\s]?17\b|'
    r'\bfeminino\b|'
    r'\bfem\b|'
    r'\bwomen\b|'
    r'\sF\b|'
    r'\sII\b|'
    r'\s2\b|'
    r'\bap\b|'
    r'\bpa\b'
    r')',
    re.IGNORECASE
)

In [6]:
# ============== processos de apostas ===================
def Eventos_do_Dia(driver):
    esperar(2)
    
    loc = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, "container-main-left"))
    )
    scroll_humano(driver,direcao="descer")

    botao = WebDriverWait(loc, 10).until(
        EC.element_to_be_clickable((By.XPATH, './/a[contains(@href,"/ptb/bet/today-events/soccer")]'))
    )

    mover_mouse(driver, botao)
    esperar()

    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", botao)
    esperar(0.5, 1.5)

    botao.click()
    esperar(1.5, 3.5)

def entrar_no_jogo(driver, nome, wait):

    debug(f"🔍 Procurando: {nome}", tipo="OK")

    xpath_time = f'.//div[normalize-space()="{nome}"]'

    wait.until(
        EC.presence_of_element_located(
            (By.CSS_SELECTOR, ".fixture-body")
        )
    )

    tentativas_scroll = 0
    max_scroll = 19

    while tentativas_scroll <= max_scroll:

        jogos = driver.find_elements(By.CSS_SELECTOR, ".fixture-body")

        for i in range(len(jogos)):

            try:
                jogos = driver.find_elements(
                    By.CSS_SELECTOR,
                    ".fixture-body"
                )

                jogo = jogos[i]

                # procura o time dentro do jogo
                try:
                    elemento_time = jogo.find_element(
                        By.XPATH,
                        xpath_time
                    )

                except NoSuchElementException:
                    continue

                # scroll só se necessário
                if not elemento_time.is_displayed():

                    driver.execute_script(
                        """
                        arguments[0].scrollIntoView({
                            block: 'center'
                        });
                        """,
                        elemento_time
                    )

                    esperar(0.4, 0.9)

                # pega o link do jogo
                link = jogo.find_element(By.XPATH, ".//a")

                mover_mouse(driver, link)

                esperar(0.3, 0.8)

                driver.execute_script(
                    "arguments[0].click();",
                    link
                )

                debug("✅ Cliquei no jogo", tipo="OK")

                return True

            except StaleElementReferenceException:

                debug(
                    "🔄 Stale detectado, reprocessando...",
                    tipo="ERRO"
                )

                continue

            except Exception as e:

                debug(
                    f"⚠️ Erro inesperado: {e}",
                    tipo="ERRO"
                )

                continue

        # ========= VERIFICA FINAL =========
        final = driver.find_elements(By.XPATH, XPATH_FINAL)

        if final:

            debug(
                "🏁 Final da página detectado",
                tipo="OCUTOS"
            )

            break

        # ========= SCROLL =========
        tentativas_scroll += 1

        debug_linha(
            f"⬇️ Scroll leve tentativa {tentativas_scroll}",
            tipo="ALERTA"
        )

        driver.execute_script(
            "window.scrollBy(0, 300);"
        )

        esperar(0.5, 1.0)

    debug("❌ Não achou esse jogo", tipo="ERRO")

    return False

def verificar_dados(driver, wait):
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
    esperar(1.5, 3.0)

    erro = driver.find_elements(By.CSS_SELECTOR, '[data-test="message-box-error-msg"]')

    if erro:
        debug("⚠️ Sem dados, voltando...",tipo="ALERTA")

        driver.back()
        esperar(1.0, 2.5)

        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".fixture-body")))
        return True

    return False

def ambas_sim(driver, wait):
    try:
        botao_sim = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, '//a[.//span[normalize-space()="Sim"]]')
            )
        )

        mover_mouse(driver, botao_sim)
        esperar(0.3, 1.0)

        driver.execute_script("arguments[0].click();", botao_sim)
        debug("✔️ Cliquei em SIM",tipo="OK")

    except Exception as e:
        debug(f"⚠️ Botão SIM não encontrado o erro e {e}",tipo="ERRO")
            
def inserir_valor(driver, wait, valor_aposta="1"):
    try:
        # Modo teste → não aposta
        if not valor_aposta:
            debug("🧪 Modo teste ativado (sem aposta)", tipo="INFO")
            return False

        valor = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, '//input[@placeholder="Montante"]')
            )
        )

        mover_mouse(driver, valor)
        esperar()

        valor.clear()

        for n in str(valor_aposta):
            valor.send_keys(n)
            time.sleep(random.uniform(0.05, 0.2))

        debug("💰 Valor inserido", tipo="OK")

    except Exception as e:
        debug(f"⚠ Não achei o campo do valor: {e}", tipo="ERRO")
        pass

def confirmar_aposta(driver, wait):
    try:
        botao_apostar = wait.until(
            EC.element_to_be_clickable((By.ID, "betslip-placebet-btn"))
        )

        mover_mouse(driver, botao_apostar)
        botao_apostar.click()

        debug("🚀 realizando aposta",tipo="OK")
        esperar(1.0, 2.0)
    except Exception as e:
        debug(f"erro nao achei o botao o erro e {e}",tipo="ERRO")

def limpar_bilhete(driver, wait):
    try:
        botao_limpar = wait.until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 'li.clear-bet-slip a'))
        )

        mover_mouse(driver, botao_limpar)
        esperar()

        botao_limpar.click()
        esperar(2.0, 4.0)
        debug("🧹 Bilhete limpo",tipo="OK")
    except Exception as e:
        debug(f"nao achei o botao limpa o erro {e}",tipo="ERRO")
               
def voltar_lista(driver, wait):
    try:
        driver.back()
        esperar(1.0, 2.5)

        wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".fixture-body"))
        )
        debug("⏮ retornando.....✅",tipo="INFO")
    except Exception as e:
        debug(f"nao conseguir volta o erro {e}",tipo="ERRO")

def finalizar(driver, wait):

    try:

        scroll_humano(driver, direcao="subir")

        esperar(2, 3)

        driver.execute_script("window.stop();")

        botao_logo = wait.until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, 'a[data-test="header-logo"]')
            )
        )

        mover_mouse(driver, botao_logo)

        esperar()

        driver.execute_script(
            "arguments[0].click();",
            botao_logo
        )

        

    except Exception as e:

        debug(
            f"⚠️ Não consegui finalizar | erro: {type(e).__name__}",
            tipo="ERRO"
        )

def rodar_Zenoh(driver, wait, times2, valor_aposta="1"):

    if not times2:
        debug("⚠ Nenhum jogo disponível para Fase 2", tipo="INFO")
        return

    debug(f"🔁 segunda faze : {ZENOH} 🤖", tipo="INFO")

    try:
        Eventos_do_Dia(driver)
        driver.refresh()
        esperar(1,5)
        for nome in times2:

            try:
                separador()
                debug(f"🎯 Processando jogo: {nome}", tipo="INFO")

                if not entrar_no_jogo(driver, nome, wait):
                    debug(f"⏭ Pulando jogo: {nome}", tipo="ALERTA")
                    separador()
                    continue
                esperar(0.1,0.8)

                verificar_dados(driver, wait)
                esperar(0.1,0.9)

                ambas_sim(driver, wait)
                esperar(0.1,0.5)

                inserir_valor(driver, wait, valor_aposta)
                esperar(0.1,0.5)
                    
                if inserir_valor(driver,wait,valor_aposta):
                    confirmar_aposta(driver, wait)
                    esperar(0.1,0.5)


                limpar_bilhete(driver, wait)
                esperar(0.1,0.5)

                voltar_lista(driver, wait)

                debug(f"✅ Jogo finalizado: {nome}", tipo="OK")

            except Exception as erro_jogo:

                debug(
                    f"⚠ Erro no jogo {nome}: {erro_jogo}",
                    tipo="ALERTA"
                )

                # tenta voltar pro estado inicial
                try:
                    limpar_bilhete(driver, wait)
                    voltar_lista(driver, wait)
                except:
                    pass

                continue
        finalizar(driver, wait)

    except Exception as erro:

        driver.save_screenshot("erro_fase2.png")

        debug(f"tipo de erro {type(erro)}", tipo="INFO")
        debug(f"repr erro {repr(erro)}", tipo="INFO")
        debug(f"erro real {erro}", tipo="INFO") 

    finally:

        debug(f"🛑 Finalizando {ZENOH}", tipo="OK")

        
        


In [ ]:
#sistema
separador()
#⚡ ZEN'OH ⚡ 

debug(f"🚀 Iniciando sistema!! {ZENOH}🤖", tipo="INFO")
separador()

navegador = iniciar_driver()

abrir_sites(navegador, url1, url2)
esperar(2)

debug("🔐 Realizando autenticação", tipo="INFO")
esperar_pagina(navegador,tentativas=3)
login_se_necessario(navegador)

limpar_popups(navegador)
esperar(2)
separador()

# =========================
# FASE 1
# =========================
debug("📡 FASE 1 - Coleta e organização de jogos", tipo="INFO")

ir_para_aba(navegador, 1)
esperar(2)
separador()

favoritar(navegador, times)
separador()
esperar(5)

limpar_favoritos(navegador)
esperar(5)
resultados = pegar_times_hoje(navegador, times)

interno = resultados["jogos_internos"]
times2 = resultados["jogos_externos"]


# =========================
# FASE 2
# =========================
separador()
ir_para_aba(navegador, 2)

wait = WebDriverWait(navegador, 30)
esperar(5)
separador()

rodar_Zenoh(navegador, wait, times2, "")


════════════════════════════════════════════════════════════

[22:47:56] [INFO] 🚀 Iniciando sistema!! ⚡ ZEN'OH ⚡🤖

════════════════════════════════════════════════════════════


════════════════════════════════════════════════════════════

[22:48:11] [INFO] 🌐 Abrindo página n-1 >>>>: https://www.flashscore.com.br/
[22:48:43] [OK] ✅ Página carregada!

════════════════════════════════════════════════════════════

[22:48:43] [INFO] 🌐 Abrindo página n-2 >>>>: https://esportesdasorte.bet.br/ptb/bet/main
[22:48:54] [OK] ✅ Página carregada!

════════════════════════════════════════════════════════════

[22:49:01] [INFO] 🔐 Realizando autenticação
[22:49:05] [INFO] ⏳ Verificando status do login... aguarde
[22:49:36] [OK] 🔐 Fazendo login...
[22:49:40] [OK] 🔐 Login realizado com sucesso!

════════════════════════════════════════════════════════════

[22:50:00] [INFO] 📡 FASE 1 - Coleta e organização de jogos
[22:50:00] [INFO] 🔄 Mudou para aba 1
[22:50:00] [INFO] 🌐 URL atual: https://www.flashscor

In [10]:
ir_para_aba(navegador, 1)
pegar = pegar_times_hoje(navegador, times)
display(pegar['jogos_internos'] )
display(pegar['jogos_externos'] )

[20:36:33] [INFO] 🔄 Mudou para aba 1
[20:36:33] [INFO] 🌐 URL atual: https://www.flashscore.com.br/favoritos/
[20:36:34] [INFO] 📦 Jogos internos encontrados: 3
[20:36:34] [INFO] 🌎 Jogos externos encontrados: 5


[{'casa': 'cruzeiro', 'fora': 'chapecoense'},
 {'casa': 'corinthians', 'fora': 'atletico-mg'},
 {'casa': 'vasco', 'fora': 'red bull bragantino'}]

['Arsenal', 'Remo', 'Ponte Preta', 'CSD Macará', 'Criciúma']

In [ ]:
ir_para_aba(navegador, 2)
testar_xpath(navegador,"//div[contains(@class,'modul-accordion')]")

In [33]:
#marcadores
mercados_dict = {}

# pega todos os mercados
mercados = navegador.find_elements(
    By.XPATH,
    "//div[contains(@class,'bet-type-group')]"
)

for mercado in mercados:

    try:
        # titulo do mercado
        titulo = mercado.find_element(
            By.XPATH,
            ".//span[contains(@class,'header-text')]"
        ).text.strip()

        # cria subdic
        mercados_dict[titulo] = {}

        # pega todos os botoes
        botoes = mercado.find_elements(
            By.XPATH,
            ".//a[contains(@class,'bet-btn')]"
        )

        for botao in botoes:

            nome = botao.find_element(
                By.CLASS_NAME,
                "bet-btn-text"
            ).text.strip()

            odd = botao.find_element(
                By.CLASS_NAME,
                "bet-btn-odd"
            ).text.strip()

            mercados_dict[titulo][nome] = odd

    except Exception as erro:
        print("erro:", erro)

print(mercados_dict)

{'Resultado': {'Stars FC': '5.54', 'Empate': '4.81', 'Redlands FC': '1.43'}, 'Total Gols': {'Mais de 1.5': '1.10', 'Menos de 1.5': '5.65', 'Mais de 2.5': '1.35', 'Menos de 2.5': '2.88', 'Mais de 3.5': '1.87', 'Menos de 3.5': '1.80', 'Mais de 4.5': '2.90', 'Menos de 4.5': '1.34', 'Mais de 5.5': '4.77', 'Menos de 5.5': '1.14'}, 'Ambas equipes marcam': {'Sim': '1.50', 'Não': '2.37'}, 'Dupla chance': {'Casa Ou Empate': '2.60', 'Casa Ou Fora': '1.18', 'Empate Ou Fora': '1.15'}, 'Empate sem aposta': {'Stars FC': '4.08', 'Redlands FC': '1.17'}, 'Handicap': {'Casa (1:0)': '2.63', 'Empate (1:0)': '4.05', 'Fora (1:0)': '2.03'}, 'Handicap Asiatico': {'Casa (+0.75)': '2.35', 'Fora (-0.75)': '1.50', 'Casa (+1.0)': '2.10', 'Fora (-1.0)': '1.63', 'Casa (+1.25)': '1.84', 'Fora (-1.25)': '1.83', 'Casa (+1.5)': '1.67', 'Fora (-1.5)': '2.03', 'Casa (+1.75)': '1.53', 'Fora (-1.75)': '2.30'}, '1ª tempo - Handicap Asiatico': {'Casa (+0.25)': '2.27', 'Fora (-0.25)': '1.52', 'Casa (+0.5)': '1.86', 'Fora (-0.5

In [34]:
display(mercados_dict)

{'Resultado': {'Stars FC': '5.54', 'Empate': '4.81', 'Redlands FC': '1.43'},
 'Total Gols': {'Mais de 1.5': '1.10',
  'Menos de 1.5': '5.65',
  'Mais de 2.5': '1.35',
  'Menos de 2.5': '2.88',
  'Mais de 3.5': '1.87',
  'Menos de 3.5': '1.80',
  'Mais de 4.5': '2.90',
  'Menos de 4.5': '1.34',
  'Mais de 5.5': '4.77',
  'Menos de 5.5': '1.14'},
 'Ambas equipes marcam': {'Sim': '1.50', 'Não': '2.37'},
 'Dupla chance': {'Casa Ou Empate': '2.60',
  'Casa Ou Fora': '1.18',
  'Empate Ou Fora': '1.15'},
 'Empate sem aposta': {'Stars FC': '4.08', 'Redlands FC': '1.17'},
 'Handicap': {'Casa (1:0)': '2.63',
  'Empate (1:0)': '4.05',
  'Fora (1:0)': '2.03'},
 'Handicap Asiatico': {'Casa (+0.75)': '2.35',
  'Fora (-0.75)': '1.50',
  'Casa (+1.0)': '2.10',
  'Fora (-1.0)': '1.63',
  'Casa (+1.25)': '1.84',
  'Fora (-1.25)': '1.83',
  'Casa (+1.5)': '1.67',
  'Fora (-1.5)': '2.03',
  'Casa (+1.75)': '1.53',
  'Fora (-1.75)': '2.30'},
 '1ª tempo - Handicap Asiatico': {'Casa (+0.25)': '2.27',
  'Fora